In [1]:
import os
import re
import json
import pickle
import random
import torch
import string
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from copy import deepcopy
from tqdm import tqdm
from collections import defaultdict, Counter
from transformers import AutoTokenizer

In [2]:
def pickle_load(path):
    with open(path, "rb") as f:
        data = pickle.load(f)
    return data

def pickle_dump(path, data):
    with open(path, "wb") as f:
        pickle.dump(data, f)

def json_load(path):
    with open(path, mode='r', encoding='utf-8') as f:
        data = json.load(f)
    return data

def json_dump(path, data):
    with open(path, mode='w', encoding='utf-8') as f:
        json.dump(data, f, indent=4, ensure_ascii=False)

def jsonl_load(path):
    data = []
    with open(path, mode='r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line))
    return data

def jsonl_dump(path, data):
    with open(path, mode='w', encoding='utf-8') as f:
        for qd in data:
            f.write(f"{json.dumps(qd, ensure_ascii=False)}\n")

In [3]:
data = jsonl_load('/data/FIRST/datasets/rank_zephyr_training_data_alpha.jsonl')

In [4]:
len(data)

39912

In [23]:
for d in data:
    print(d.keys())
    print(d['id'])
    print(len(d['conversations']))
    print(d['conversations'][0])
    print(d['conversations'][2])
    print(d['conversations'][1].keys())
    print(d['conversations'][1]['from'])
    print(d['conversations'][1]['value'][:1024])
    print('-----')
    print(d['conversations'][1]['value'][-1024:])
    break

dict_keys(['id', 'conversations'])
identity_1183415_da0open_ai_ada2random_ct0
3
{'from': 'system', 'value': 'You are RankLLM, an intelligent assistant that can rank passages based on their relevancy to the query.'}
{'from': 'gpt', 'value': '[C] > [H] > [N] > [O] > [T] > [Q] > [P] > [F] > [I] > [B] > [A] > [M] > [J] > [R] > [G] > [K] > [D] > [E] > [L] > [S]'}
dict_keys(['from', 'value'])
human
I will provide you with 20 passages, each indicated by a alphabetic identifier []. Rank the passages based on their relevance to the search query: fibroblasts synthesize what of the ground substance.

[A] The ground substance secreted by a fibroblast has a lot to do with the type of connective tissue it produces, too. Bone, cartilage, fat, and blood are all special forms of connective tissue. What makes them radically different is the composition of their ground substance.
[B] The ground substance secreted by a fibroblast has a lot to do with the type of connective tissue it produces, too. Bone, c

## rewritten docs to cache - conditional

In [86]:
def extract_contents(text):
    if 'Step 1. Document Rewriting:' not in text:
        return ""
    if 'Step 2. Answer:' not in text:
        return ""
    text = text.split('Step 1. Document Rewriting:')[-1]
    text = text.split('Step 2. Answer:')[0]
    text = text.strip()
    return text

def extract_answer(text):
    if 'Step 2. Answer:' not in text:
        return ""
    text = text.split('Step 2. Answer:')[-1]
    text = text.strip()
    return text

def detect_invalid_rewritten_doc(text):
    text = text.lower()
    if 'no_rewrite' in text:
        return True
    if 'no rewrite' in text:
        return True
    return False

def normalize_answer_qa(s):
    def remove_articles(text):
        return re.sub(r"\b(a|an|the)\b", " ", text)
    def white_space_fix(text):
        return " ".join(text.strip().split())
    def remove_punc(text):
        exclude = set(string.punctuation)
        return "".join(ch for ch in text if ch not in exclude)
    def lower(text):
        return text.lower()
    return white_space_fix(remove_articles(remove_punc(lower(s))))

In [87]:
path = '/data/Search-o1/data/QA_Datasets/msmarco'
qa_name = 'msmarco_train_abs_full.json'
qa_list = json_load(os.path.join(path, qa_name))
question2answers = {qa['Question']:qa['answer'] for qa in qa_list}
len(qa_list), len(question2answers)

(18566, 18566)

In [88]:
path = '/data/Search-o1/cache/msmarco'
cache_name = 'search_cache_train_abs_full.json'
raw_cache = json_load(os.path.join(path, cache_name))

default_cache_name = 'search_cache_train_abs_full_conditional_cot2_rewritten_snowflake-llama-3.3-70b'
cache_name_list = [
    f'{default_cache_name}_0.json',
    f'{default_cache_name}_5000.json'
]
cache = {}
for cache_name in cache_name_list:
    tmp_cache = json_load(os.path.join(path, cache_name))
    for k,v in tmp_cache.items():
        assert k not in cache
        cache[k] = v

len(raw_cache), len(cache)

(18566, 10000)

In [89]:
new_cache = {}

for question, docs in cache.items():
    answers = [normalize_answer_qa(answer) for answer in question2answers[question]]
    raw_docs = raw_cache[question]
    assert len(raw_docs) == len(docs)

    new_docs = []
    for i, (raw_doc, doc) in enumerate(zip(raw_docs, docs)):
        assert raw_doc['id'] == doc['id']
        contents = extract_contents(doc['contents'])
        if detect_invalid_rewritten_doc(contents):
            continue
        # pred_answer = normalize_answer_qa(extract_answer(doc['contents']))
        # flag = False
        # for answer in answers:
        #     if answer in pred_answer:
        #         flag = True
        # if not flag:
        #     continue
        new_doc = deepcopy(doc)
        new_doc['position'] = i
        new_doc['contents'] = contents
        new_docs.append(new_doc)
    if len(new_docs) == 0:
        continue
    new_cache[question] = new_docs
len(new_cache)

10000

In [90]:
path = '/data/Search-o1/cache/msmarco'
default_cache_name = 'search_cache_train_abs_full_conditional_cot2_rewritten_snowflake-llama-3.3-70b'
cache_name = f'{default_cache_name}_split_norewrite.json'
json_dump(os.path.join(path, cache_name), new_cache)

## Convert format - cache to training data

In [104]:
def get_input_prompt(document, question):
    user_prompt = (
        'You are a helpful assistant. Your job is to analyze the documents below and rewrite only the information relevant to answering the question.\n'
        'List each relevant document and its contribution. Do not include unrelated documents.\n\n'
        
        'Question:\n'
        f'{question}\n\n'
        
        'Documents:\n'
        f'{document}\n\n'
    )
    return user_prompt

### merge qa & cache

In [105]:
def detect_invalid_rewritten(text):
    text = text.lower()
    if 'no_rewrite' in text:
        return True
    if len(text) == 0:
        return True
    return False

In [106]:
dataset_name = 'msmarco'
cache_names = [
    'search_cache_train_abs_full_conditional_cot2_rewritten_snowflake-llama-3.3-70b_split_norewrite.json', 
]

new_cache = {}
for cache_name in cache_names:
    cache = json_load(f'/data/Search-o1/cache/{dataset_name}/{cache_name}')
    for question, docs in cache.items():
        flag = False
        new_docs = []
        for doc in docs:
            if flag or detect_invalid_rewritten(doc['contents']):
                doc['contents'] = '[NO_REWRITE]'
                flag = True
            new_docs.append(deepcopy(doc))
        new_cache[question] = new_docs
len(new_cache)

10000

In [95]:
qa_list = json_load(f'/data/Search-o1/data/QA_Datasets/{dataset_name}/msmarco_train_abs_full.json')
new_qa_list = []

for qa in qa_list:
    question = qa['Question']
    if question in new_cache:
        new_qa_list.append(qa)
len(new_qa_list)

10000

In [107]:
path = f'/data/Search-o1/data/QA_Datasets/{dataset_name}/msmarco_train_abs_full_conditional_cot2_norewrite.json'
# path = f'/data/Search-o1/data/QA_Datasets/{dataset_name}/msmarco_dev_abs.json'
json_dump(path, new_qa_list)

path = f'/data/Search-o1/cache/{dataset_name}/search_cache_train_abs_full_conditional_cot2_rewritten_snowflake-llama-3.3-70b_norewrite.json'
# path = f'/data/Search-o1/cache/{dataset_name}/search_cache_dev_abs_full_conditional_rewritten_snowflake-llama-3.3-70b_ranked.json'
json_dump(path, new_cache)

### cache to jsonl for training

In [108]:
# qa_name = 'msmarco_train_abs_4k.json'
# cache_name = 'search_cache_train_abs_full_conditional_rewritten_snowflake-llama-3.3-70b_ranked_4k.json'
# qa_name = 'msmarco_dev_abs.json'
# cache_name = 'search_cache_dev_abs_full_conditional_rewritten_snowflake-llama-3.3-70b_ranked.json'
qa_name = 'msmarco_train_abs_full_conditional_cot2_norewrite.json'
cache_name = 'search_cache_train_abs_full.json'

qa_list = json_load(f'/data/Search-o1/data/QA_Datasets/{dataset_name}/{qa_name}')
cache = json_load(f'/data/Search-o1/cache/{dataset_name}/{cache_name}')
# assert len(qa_list) == len(cache)
len(qa_list), len(cache)

(10000, 18566)

In [109]:
cache_name = 'search_cache_train_abs_full_conditional_cot2_rewritten_snowflake-llama-3.3-70b_norewrite.json'
rewritten_cache = json_load(f'/data/Search-o1/cache/{dataset_name}/{cache_name}')
len(rewritten_cache)

10000

In [112]:
data = []

for qa in qa_list:
    qid = qa['id']
    question = qa['Question']
    answers = qa['answer']
    docs = cache[question]

    document = ""
    for i, doc in enumerate(docs):
        document += f"Document {i + 1}:\n"
        document += f"{doc.get('contents', '')}\n\n"

    input_prompt = get_input_prompt(document, question)

    rewritten_docs = rewritten_cache[question]
    rewritten_document = ""
    rewritten_document += '<rewritten_docs>\n'
    for doc in rewritten_docs:
        position = doc['position']
        rewritten_document += f"Document {position + 1}:\n"
        rewritten_document += f"{doc['contents']}\n\n"
    rewritten_document += '</rewritten_docs>\n'

    # assert len(rewritten_docs) == 1
    # rewritten_document = rewritten_docs[0]['contents']
    # rewritten_document = ""
    # for i, doc in enumerate(rewritten_docs):
    #     rewritten_document += f"Document {i + 1}:\n"
    #     rewritten_document += f"{doc.get('contents', '')}\n\n"
    # rewritten_document += '</doc_analysis>\n'

    data.append({
        'id': qid,
        'conversations': [
            {'from': 'system', 'value': ''}, 
            {'from': 'human', 'id':qid, 'value': input_prompt}, 
            {'from': 'llama-3.3-70B-Instruct', 'value': rewritten_document}, 
        ]
    })
len(data)

10000

In [42]:
# data = []

# for qa in qa_list:
#     qid = qa['id']
#     question = qa['Question']
#     answers = qa['answer']
#     docs = cache[question]
    
#     for i, doc in enumerate(docs):
#         other_docs = docs[:i] + docs[i+1:]
#         other_documents = ''
#         for j, other_doc in enumerate(other_docs):
#             other_documents += f"Document {j + 1}:\n"
#             other_documents += f"{other_doc.get('contents', '')}\n\n"

#         docid = doc['id']
#         document = doc['contents']
#         input_prompt = conditional_prompt(document, question, other_documents)

#         data.append({
#             'id': '-'.join([str(qid), str(docid)]),
#             'conversations': [
#                 {'from': 'system', 'value': ''}, 
#                 {'from': 'human', 'id':docid, 'value': input_prompt}, 
#                 {'from': 'llama-3.3-70B-Instruct', 'value': document}, 
#             ]
#         })
# len(data)

9985

In [113]:
path = f'/data/FIRST/datasets/{dataset_name}_train_conditional_cot2_norewrite.jsonl'
jsonl_dump(path, data)

In [114]:
path = f'/data/FIRST/datasets/{dataset_name}_dev_conditional_cot2_norewrite.jsonl'
jsonl_dump(path, data[:100])

In [10]:
dataset_name = 'msmarco'
cache_names = [
    'search_cache_train_abs_full_conditional_cot2_rewritten_snowflake-llama-3.3-70b_0.json', 
    'search_cache_train_abs_full_conditional_cot2_rewritten_snowflake-llama-3.3-70b_5000.json', 
    'search_cache_train_abs_full_conditional_cot2_rewritten_snowflake-llama-3.3-70b_10000.json', 
    'search_cache_train_abs_full_conditional_cot2_rewritten_snowflake-llama-3.3-70b_15000.json', 
    'search_cache_train_ext_full_conditional_cot2_rewritten_snowflake-llama-3.3-70b_0.json', 
    'search_cache_train_ext_full_conditional_cot2_rewritten_snowflake-llama-3.3-70b_6300.json', 
    'search_cache_train_ext_full_conditional_cot2_rewritten_snowflake-llama-3.3-70b_12600.json', 
]

new_cache = {}
for cache_name in cache_names:
    cache = json_load(f'/data/Search-o1/cache/{dataset_name}/{cache_name}')
    for question, docs in cache.items():
        new_cache[question] = docs
len(new_cache)

37466

In [11]:
cache_name = 'search_cache_train_full_conditional_cot2_rewritten_snowflake-llama-3.3-70b.json'
json_dump(f'/data/Search-o1/cache/{dataset_name}/{cache_name}', new_cache)

In [14]:
dataset_name = 'msmarco'
cache_names = ['search_cache_train_abs_full.json', 'search_cache_train_ext_full.json']

new_raw_cacahe = {}
for cache_name in cache_names:
    cache = json_load(f'/data/Search-o1/cache/{dataset_name}/{cache_name}')
    print(f'{cache_name}: {len(cache)}')
    for question, docs in cache.items():
        new_raw_cacahe[question] = docs

print(len(new_raw_cacahe))
new_raw_cacahe = {k:v for k,v in new_raw_cacahe.items() if k in new_cache}
print(len(new_raw_cacahe))

search_cache_train_abs_full.json: 18566
search_cache_train_ext_full.json: 125524
144090
37466


In [15]:
cache_name = 'search_cache_train_full.json'
json_dump(f'/data/Search-o1/cache/{dataset_name}/{cache_name}', new_raw_cacahe)

In [16]:
dataset_name = 'msmarco'
qa_names = ['msmarco_train_abs_full.json', 'msmarco_train_ext_full.json']

new_qa_list = []
qids = set()
for qa_name in qa_names:
    qa_list = json_load(f'/data/Search-o1/data/QA_Datasets/{dataset_name}/{qa_name}')
    print(f'{qa_name}: {len(qa_list)}')
    new_qa_list.extend(qa_list)
    for qa in qa_list:
        qids.add(qa['id'])

print(len(new_qa_list))
new_qa_list = [qa for qa in new_qa_list if qa['Question'] in new_cache]
print(len(new_qa_list), len(qids))

msmarco_train_abs_full.json: 18566
msmarco_train_ext_full.json: 125524
144090
37466 144090


In [17]:
qa_name = 'msmarco_train_full.json'
json_dump(f'/data/Search-o1/data/QA_Datasets/{dataset_name}/{qa_name}', new_qa_list)